<a href="https://colab.research.google.com/github/shiosabax/test_web_program/blob/main/kabuka_sihyou.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import time
import requests
import json

# 1. 取得したい主要株価指数のシンボル（識別子）を「リスト」で定義
# ^N225: 日経平均株価, ^TOPX: TOPIX, ^GSPC: S&P500, ^IXIC: NASDAQ総合
indices = ["^N225", "^TOPX", "^GSPC", "^IXIC"]

# サーバーにブラウザからのアクセスであることを伝えるヘッダー情報
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("=== 主要株価指数のデータ取得開始 ===")

# 2. リストの中身を for ループで1つずつ順番に処理
for symbol in indices:
    # データを取得するURL（Yahoo Financeの公開エンドポイント形式）
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}?interval=1d&range=1d"

    # requestsを使ってデータを取得
    response = requests.get(url, headers=headers)

    # 通信が成功（ステータスコード 200）したか確認
    if response.status_code == 200:
        data = response.json()

        try:
            meta = data["chart"]["result"][0]["meta"]
            name = meta.get("shortName", symbol)
            currency = meta.get("currency", "")
            current_price = meta.get("regularMarketPrice", 0)
            prev_close = meta.get("chartPreviousClose", current_price)

            # 前日比と変動率の計算
            diff = current_price - prev_close
            diff_percent = (diff / prev_close) * 100 if prev_close else 0

            # 結果を分かりやすく出力
            sign = "+" if diff >= 0 else ""
            print(f"【{name} ({symbol})】")
            print(f"  最新値: {current_price:,.2f} {currency}")
            print(f"  前日比: {sign}{diff:,.2f} ({sign}{diff_percent:.2f}%)\n")

        except (KeyError, IndexError) as e:
            print(f"【{symbol}】データの解析に失敗しました。")
    else:
        print(f"【{symbol}】データの取得に失敗しました（ステータス: {response.status_code}）")

    # サーバー負荷軽減のため1秒待機
    time.sleep(1)

print("=== 取得完了 ===")

=== 主要株価指数のデータ取得開始 ===
【Nikkei 225 (^N225)】
  最新値: 66,215.34 JPY
  前日比: -96.59 (-0.15%)

【^TOPX (^TOPX)】
  最新値: 0.00 None
  前日比: +0.00 (+0.00%)

【S&P 500 (^GSPC)】
  最新値: 7,635.94 USD
  前日比: -50.20 (-0.65%)

【NASDAQ Composite (^IXIC)】
  最新値: 26,142.28 USD
  前日比: -228.61 (-0.87%)

=== 取得完了 ===


In [3]:
import time
import requests
import pandas as pd
from google.colab import files

indices = ["^N225", "^TOPX", "^GSPC", "^IXIC"]
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 取得したデータをためておく空のリストを用意
data_list = []

print("=== 主要株価指数のデータ取得中... ===")

for symbol in indices:
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}?interval=1d&range=1d"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        try:
            meta = data["chart"]["result"][0]["meta"]
            name = meta.get("shortName", symbol)
            currency = meta.get("currency", "")
            current_price = meta.get("regularMarketPrice", 0)
            prev_close = meta.get("chartPreviousClose", current_price)

            diff = current_price - prev_close
            diff_percent = (diff / prev_close) * 100 if prev_close else 0

            # 1件分のデータを辞書形式にしてリストに追加 (append)
            data_list.append({
                "シンボル": symbol,
                "指数名": name,
                "最新値": round(current_price, 2),
                "通貨": currency,
                "前日比": round(diff, 2),
                "変動率(%)": round(diff_percent, 2)
            })
        except (KeyError, IndexError):
            pass

    time.sleep(1)

# リストからデータフレーム（表形式）を作成
df = pd.DataFrame(data_list)
print("\n--- 取得結果の一覧 ---")
print(df)

# CSVファイルとして書き出し
csv_filename = "stock_indices.csv"
df.to_csv(csv_filename, index=False, encoding="utf_8_sig")
print(f"\n『{csv_filename}』として保存しました。")

# Google Colabからパソコンへ直接ダウンロード（必要に応じて実行）
# files.download(csv_filename)

=== 主要株価指数のデータ取得中... ===

--- 取得結果の一覧 ---
    シンボル               指数名       最新値    通貨     前日比  変動率(%)
0  ^N225        Nikkei 225  66215.34   JPY  -96.59   -0.15
1  ^TOPX             ^TOPX      0.00  None    0.00    0.00
2  ^GSPC           S&P 500   7636.90   USD  -49.24   -0.64
3  ^IXIC  NASDAQ Composite  26143.75   USD -227.14   -0.86

『stock_indices.csv』として保存しました。


In [ ]:
#┌────────────────────────────────────────────────────────┐
#│ 📈 主要株価指数 リアルタイム監視＆LINE通知ダッシュボード      │
#├────────────────────────────────────────────────────────┤
#│ 【サイドバー（設定エリア）】                                  │
#│ ・LINE Access Token / User ID の入力欄                 │
#│ ・アラートしきい値スライダー（例: ±1.0%）                  │
#│ ・監視対象銘柄の選択チェックボックス                           │
#│                                                        │
#│ 【メイン画面】                                          │
#│  [ 🔄 最新データを取得＆監視実行 ] ボタン                │
#│                                                        │
#│  ▼ 指標カード（日経平均 / TOPIX / S&P500 / NASDAQ）   │
#│  ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐   │
#│  │ 日経平均  │ │  TOPIX   │ │  S&P500  │ │  NASDAQ  │   │
#│  │ 38,500円 │ │ 2,700pt  │ │ 5,600pt  │ │ 17,800pt │   │
#│  │ (+1.2%)  │ │ (-0.3%)  │ │ (+0.8%)  │ │ (+1.5%)  │   │
#│  └──────────┘ └──────────┘ └──────────┘ └──────────┘   │
#│                                                        │
#│  ▼ 取得結果データテーブル（表形式）                       │
#│  ▼ [ 📥 CSVダウンロード ] ボタン                        │
#│  ▼ アラート発生ログ（LINE送信結果表示）                   │
#└────────────────────────────────────────────────────────┘

In [4]:
import time
import requests
import pandas as pd

# 1. 監視対象のインデックスとアラート条件
indices = ["^N225", "^TOPX", "^GSPC", "^IXIC"]
ALERT_THRESHOLD_PERCENT = 1.0  # 変動率が ±1.0% 以上でアラート対象とする

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

alert_items = []
print("=== 株価指数の監視・判定を実行中... ===")

for symbol in indices:
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}?interval=1d&range=1d"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        try:
            meta = data["chart"]["result"][0]["meta"]
            name = meta.get("shortName", symbol)
            currency = meta.get("currency", "")
            current_price = meta.get("regularMarketPrice", 0)
            prev_close = meta.get("chartPreviousClose", current_price)

            diff = current_price - prev_close
            diff_percent = (diff / prev_close) * 100 if prev_close else 0

            # 2. 条件分岐（absで絶対値を取り、上昇・下落どちらも判定）
            if abs(diff_percent) >= ALERT_THRESHOLD_PERCENT:
                sign = "+" if diff >= 0 else ""
                alert_items.append(
                    f"⚠️【{name}】急変動検知！\n"
                    f"現在値: {current_price:,.2f} {currency} ({sign}{diff_percent:.2f}%)"
                )
        except (KeyError, IndexError):
            pass

    time.sleep(1)

# 3. アラートの送信（通知文の作成）
if alert_items:
    message = "【市場アラート速報】\n" + "\n\n".join(alert_items)
    print("\n" + "="*30)
    print(message)
    print("="*30)

    # 外部通知を送る場合はここにWebhook送信処理を追加
    # send_discord_notify(message)
else:
    print(f"\n平常運転です（すべての指数が ±{ALERT_THRESHOLD_PERCENT}% 以内）")

=== 株価指数の監視・判定を実行中... ===

平常運転です（すべての指数が ±1.0% 以内）


In [5]:
import time
import requests

# 1. LINE Messaging API の認証情報（ご自身のものに書き換えてください）
LINE_ACCESS_TOKEN = "gDZb2T5sRD7ezm6DFM2J/DhfcS2pCS2vpljqdFVFDeyoNVSpTs/DWY85duGsjHPEYCPZNzA+AYw8VKzhx9R/wg D8E8BITV9aX05v9LGwFw7x5n1h1XKIBwLtchUGJdYkhWIy3vRsLIYKn9H8JHmN6QdB04t89/1O/w1cDnyilFU=$0"
USER_ID = "Ucd59d2c6345de62acb33d0a9ffc52d2e"

# 2. 監視銘柄としきい値（±1.0%以上の変動で検知）
indices = ["^N225", "^TOPX", "^GSPC", "^IXIC"]
ALERT_THRESHOLD_PERCENT = 1.0

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# --- LINE送信関数 ---
def send_line_message(token, user_id, message_text):
    url = "https://api.line.me/v2/bot/message/push"
    auth_header = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }
    payload = {
        "to": user_id,
        "messages": [
            {
                "type": "text",
                "text": message_text
            }
        ]
    }
    res = requests.post(url, headers=auth_header, json=payload)
    if res.status_code == 200:
        print("✅ LINE通知を正常に送信しました！")
    else:
        print(f"❌ LINE送信エラー: {res.status_code}, {res.text}")

# --- 株価取得＆判定 ---
alert_items = []
print("=== 株価指数の監視・判定中... ===")

for symbol in indices:
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}?interval=1d&range=1d"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        try:
            meta = data["chart"]["result"][0]["meta"]
            name = meta.get("shortName", symbol)
            currency = meta.get("currency", "")
            current_price = meta.get("regularMarketPrice", 0)
            prev_close = meta.get("chartPreviousClose", current_price)

            diff = current_price - prev_close
            diff_percent = (diff / prev_close) * 100 if prev_close else 0

            # 条件に合致した場合にメッセージを作成
            if abs(diff_percent) >= ALERT_THRESHOLD_PERCENT:
                sign = "+" if diff >= 0 else ""
                alert_items.append(f"・{name} ({symbol})\n  現在値: {current_price:,.2f} {currency}\n  変動: {sign}{diff_percent:.2f}%")
        except (KeyError, IndexError):
            pass

    time.sleep(1)

# --- 通知の実行 ---
if alert_items:
    alert_message = "⚠️【市場アラート速報】\n\n" + "\n\n".join(alert_items)
    send_line_message(LINE_ACCESS_TOKEN, USER_ID, alert_message)
else:
    print(f"平常運転です（全指数 ±{ALERT_THRESHOLD_PERCENT}% 以内）")

=== 株価指数の監視・判定中... ===
平常運転です（全指数 ±1.0% 以内）


In [6]:
import time
import requests

!pip install streamlit pandas requests -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 94.1 MB/s eta 0:00:00


In [7]:
%%writefile app.py
import streamlit as st
import requests
import pandas as pd
import time

# --- ページ設定 ---
st.set_page_config(page_title="株価指数モニター", page_icon="📈", layout="wide")

st.title("📈 主要株価指数 監視＆LINEアラート通知アプリ")
st.caption("最新の株価指数を取得し、設定したしきい値を超えた場合にLINEへ通知を送ります。")

# --- サイドバー設定 ---
st.sidebar.header("⚙️ アプリ設定")
token_input = st.sidebar.text_input("LINE Channel Access Token", type="password")
user_id_input = st.sidebar.text_input("LINE User ID", placeholder="Uから始まる文字列")
threshold = st.sidebar.slider("急変動アラートのしきい値 (±%)", min_value=0.1, max_value=5.0, value=1.0, step=0.1)

# 監視対象銘柄の選択
default_indices = {
    "日経平均株価": "^N225",
    "TOPIX": "^TOPX",
    "S&P 500": "^GSPC",
    "NASDAQ 総合": "^IXIC"
}
selected_names = st.sidebar.multiselect("監視対象指数", list(default_indices.keys()), default=list(default_indices.keys()))

# --- LINE送信関数 ---
def send_line_message(token, user_id, message_text):
    if not token or not user_id:
        return False, "LINE認証情報が設定されていません。"
    url = "https://api.line.me/v2/bot/message/push"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }
    payload = {
        "to": user_id,
        "messages": [{"type": "text", "text": message_text}]
    }
    res = requests.post(url, headers=headers, json=payload)
    if res.status_code == 200:
        return True, "送信成功"
    else:
        return False, f"送信失敗: {res.status_code}"

# --- メイン処理 ---
if st.button("🔄 データを取得して監視を実行", type="primary"):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    data_list = []
    alert_items = []

    with st.spinner("各指数の最新データを取得中..."):
        cols = st.columns(len(selected_names)) if selected_names else []

        for idx, name in enumerate(selected_names):
            symbol = default_indices[name]
            url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}?interval=1d&range=1d"
            res = requests.get(url, headers=headers)

            if res.status_code == 200:
                data = res.json()
                try:
                    meta = data["chart"]["result"][0]["meta"]
                    price = meta.get("regularMarketPrice", 0)
                    prev_close = meta.get("chartPreviousClose", price)
                    currency = meta.get("currency", "")

                    diff = price - prev_close
                    diff_pct = (diff / prev_close) * 100 if prev_close else 0

                    # 画面上のメトリクスカード表示
                    with cols[idx]:
                        st.metric(
                            label=name,
                            value=f"{price:,.2f} {currency}",
                            delta=f"{diff_pct:+.2f}% ({diff:+,.2f})"
                        )

                    data_list.append({
                        "指数名": name,
                        "シンボル": symbol,
                        "最新値": round(price, 2),
                        "通貨": currency,
                        "前日比": round(diff, 2),
                        "変動率(%)": round(diff_pct, 2)
                    })

                    # しきい値判定
                    if abs(diff_pct) >= threshold:
                        sign = "+" if diff >= 0 else ""
                        alert_items.append(f"・{name}\n  最新値: {price:,.2f} {currency}\n  変動率: {sign}{diff_pct:.2f}%")
                except Exception:
                    pass
            time.sleep(0.5)

    # テーブル表示とCSVダウンロード
    if data_list:
        df = pd.DataFrame(data_list)
        st.subheader("📊 取得データ一覧")
        st.dataframe(df, use_container_width=True)

        csv_data = df.to_csv(index=False, encoding="utf_8_sig").encode("utf_8_sig")
        st.download_button(
            label="📥 CSVファイルをダウンロード",
            data=csv_data,
            file_name="market_indices.csv",
            mime="text/csv"
        )

    # アラート判定・通知処理
    st.divider()
    st.subheader("🔔 アラート判定ログ")
    if alert_items:
        msg = f"⚠️【市場急変動アラート（±{threshold}%以上）】\n\n" + "\n\n".join(alert_items)
        st.warning(f"{len(alert_items)} 件の急変動が検知されました！")
        st.text(msg)

        if token_input and user_id_input:
            success, info = send_line_message(token_input, user_id_input, msg)
            if success:
                st.success("✅ LINEへ通知を送信しました！")
            else:
                st.error(f"❌ LINE送信エラー: {info}")
        else:
            st.info("💡 LINE認証情報をサイドバーに入力すると、スマホへ通知が届きます。")
    else:
        st.success(f"全銘柄が平常運転です（すべての指数が ±{threshold}% 以内）。")

Writing app.py


In [9]:
# 1. あなたのColabインスタンスのIPアドレス（認証用パスワードになります）を確認
!curl ipv4.icanhazip.com

# 2. Streamlitアプリをバックグラウンドで起動し、localtunnelでURLを発行
#    npx localtunnel のインストール時に自動で 'yes' と答えるように '-y' オプションを追加
!streamlit run app.py & npx localtunnel --port 8501 -y

34.145.254.82
⠙

⠹⠸⠼⠴⠦Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-09-01 17:01:39.872 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.145.254.82:8501

y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://slick-years-wonder.loca.lt
  Stopping...
^C


In [ ]:
# 1. 安定してトンネル接続できる cloudflared をダウンロード
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Streamlitアプリを起動し、CloudflareのURLを発行
import subprocess
import time

# バックグラウンドでStreamlitを起動
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])

# 3秒待機
time.sleep(3)

# cloudflared で公開URLを作成
!cloudflared tunnel --url http://localhost:8501

2026-09-01T17:10:27Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-01T17:10:27Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-01T17:10:31Z INF +--------------------------------------------------------------------------------------------+
2026-09-01T17:10:31Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-01T17:10:31Z INF |  https://shirts-ocean-silver-snow.trycloudflare.com   